# How VibeVoice Works — a guided tour for LongFlow

**Estimated reading time: ~25 minutes** (mostly reading; the three code cells run on CPU in seconds).

This notebook explains the system LongFlow builds on: **VibeVoice**, Microsoft's open long-form
text-to-speech model. By the end you should be able to answer, in your own words:

1. What makes VibeVoice different from a normal language model?
2. What are its four moving parts, and what tensor shapes flow between them?
3. What exactly happens on each step of the generation loop?
4. What is classifier-free guidance, and what did *we* discover about how VibeVoice runs it?
5. Why does LongFlow keep every VibeVoice weight frozen, and where does our code attach?

Every fact here is grounded in this repo's verified sources: `docs/resources.md` (the hook map we
checked against the live model), `experiments/p0_steering/NOTES.md` (the P0 experiment log), and
`docs/negative-results.md` (what the predecessor project taught us). No model downloads, no GPU —
the code cells use plain dummy tensors.

## 1. The big picture: a language model that speaks

VibeVoice is an open (MIT-licensed) text-to-speech system from Microsoft. Its claim to fame:
it can generate up to **90 minutes of continuous audio with up to 4 distinct speakers** in a
single pass — a full podcast episode from a script. The model LongFlow uses is
`microsoft/VibeVoice-1.5B` on Hugging Face; its 64K-token context window is what buys the
90 minutes.

To see why it's interesting, start from what a normal language model (LLM) does. An LLM like
Qwen or GPT works with **tokens** — words or word-pieces from a fixed vocabulary. At each step it
looks at everything so far and picks the next token from that vocabulary. This is called
**autoregressive** (AR) generation: each output becomes part of the input for the next step.

The problem for speech: audio isn't a sequence of items from a fixed menu. It's continuous —
pressure waves, 24,000 numbers per second. Many TTS systems force audio into a discrete
vocabulary anyway ("audio codecs" with codebooks), which works but throws away detail.

VibeVoice takes a different route, from a Microsoft research idea called **LatentLM**, nicknamed
**"next-token diffusion"**. The recipe:

> Keep the autoregressive language model. But at each step, instead of picking a token from a
> vocabulary, emit a small **continuous vector** — a list of 64 real numbers that describes the
> next fraction of a second of audio. A small helper network (a *diffusion head*, explained in
> section 4) turns the LLM's internal state into that vector.

That 64-number vector is called an **acoustic latent** — "latent" just means a compressed,
learned representation that isn't directly human-readable, and "acoustic" because it encodes
sound. A separate decoder network can turn each latent back into actual audio.

So VibeVoice is, at heart, *an LLM whose "next token" is sometimes a 64-dimensional vector of
sound instead of a word*. Everything LongFlow does — the faster flow-matching head, activation
steering, speaker anchoring — bolts onto this one loop.

## 2. The anatomy: four parts

VibeVoice-1.5B is really four networks glued together.

### 2.1 The backbone — "the thoughts"

The core is **Qwen2.5-1.5B**, a stock open-source language model: **28 transformer layers**
with a **hidden size of 1536**. "Hidden size 1536" means that at every position in the sequence,
each layer passes along a vector of 1536 numbers — the model's evolving internal representation
of that position. These per-position vectors are called **hidden states**. Think of the hidden
state at the last position as the model's current "thought": everything it knows about what
should come next, compressed into 1536 numbers.

(We verified this against the live model in P0: `model.model.language_model.layers` is 28
standard `Qwen2DecoderLayer` modules, hidden size 1536. These layers are where LongFlow's
steering hooks attach.)

### 2.2 The acoustic tokenizer — ears and mouth

The **acoustic tokenizer** converts between audio and latents. It's a **σ-VAE** — a
*variational autoencoder*, which is a pair of networks: an **encoder** that squeezes audio down
into compact latent vectors, and a **decoder** that reconstructs audio from them. (The "σ"
variant fixes the amount of randomness in the encoding to a constant, which keeps the latent
space well-behaved for generation.)

The numbers that matter:

- Each latent **frame** is **64 numbers**.
- Frames come at **7.5 per second** of audio.
- Audio is 24 kHz (24,000 samples/second), so one frame stands in for 24,000 / 7.5 = **3200 raw
  audio samples** — a **3200× compression ratio**. That extreme compression is exactly what makes
  90-minute generation feasible: 90 minutes is only ~40,500 frames, which fits in an LLM context.

### 2.3 The semantic tokenizer — the "what was said" channel

A second, encoder-only network produces a **128-dimensional semantic** representation of audio —
capturing *content* (which sounds/words) more than *texture* (voice quality). It's used only in
the feedback loop (section 3): after the model generates a chunk of audio, the semantic encoder
listens to it, and that summary is fed back in as part of the next step's input.

### 2.4 The diffusion head — the thought-to-sound converter

The **diffusion head** (`VibeVoiceDiffusionHead`) is the small network that turns each thought
into a latent. We measured it live: **123.28M parameters** — small next to the 1.5B backbone but
not tiny. It's 4 layers of MLP (no attention), using a conditioning trick called **AdaLN**: the
1536-dim hidden state is projected in and used to shift/scale the head's internal activations,
so the same little network produces different sounds depending on the thought driving it.

"Diffusion" means it works by **iterative denoising**: start from pure random noise, and refine
it over several small steps into a clean 64-dim latent, guided by the conditioning vector at
every step. (For the curious: it uses v-prediction with a cosine noise schedule and a DPM-Solver++
sampler — standard modern diffusion machinery. You don't need those details to follow along.)

Let's make the shapes concrete.

In [ ]:
import torch

# The real numbers, from docs/resources.md (verified against the live model):
B = 1            # batch size
T = 12           # sequence length so far (tokens + audio frames)
d_model = 1536   # backbone hidden size  ("the thoughts")
d_latent = 64    # acoustic latent size  ("the sounds")

# 1) The backbone produces one hidden state per position:
hidden_states = torch.randn(B, T, d_model)
print("backbone output:      ", tuple(hidden_states.shape), "  [B, T, 1536]")

# 2) To generate the NEXT audio frame, only the newest thought matters:
condition = hidden_states[:, -1, :]
print("per-frame condition:  ", tuple(condition.shape), "        [B, 1536]")

# 3) The diffusion head maps (noise + condition) -> one acoustic latent.
#    A single Linear layer stands in for the real 123M-param head here:
fake_head = torch.nn.Linear(d_model + d_latent, d_latent)
noise = torch.randn(B, d_latent)
latent = fake_head(torch.cat([noise, condition], dim=-1))
print("one acoustic latent:  ", tuple(latent.shape), "          [B, 64]")

# 4) Each latent frame stands in for a big chunk of raw audio:
sample_rate, frame_rate = 24_000, 7.5
samples_per_frame = sample_rate / frame_rate
print(f"\n1 latent frame  <->  {samples_per_frame:.0f} audio samples  "
      f"({samples_per_frame/sample_rate*1000:.0f} ms of 24 kHz audio; 3200x compression)")
frames_90min = 90 * 60 * frame_rate
print(f"90 minutes of audio  =  {frames_90min:,.0f} latent frames")

That's the whole data flow in one line: **[B, T, 1536] thoughts → take the last one, [B, 1536] →
diffusion head → [B, 64] latent → decoder → 133 ms of audio.** The backbone thinks in 1536
dimensions; the system speaks in 64.

## 3. The generation loop, step by step

Now let's watch one full generation. Everything below lives in one method —
`VibeVoiceForConditionalGenerationInference.generate()` in
`vibevoice/modular/modeling_vibevoice_inference.py` (lines 327–697 in the pinned fork).

### 3.1 Building the prompt

Before any audio is generated, the processor (`VibeVoiceProcessor`) assembles one long input
sequence:

1. **A system prompt** — ordinary text tokens.
2. **Voice samples** — a few seconds of reference audio per speaker, so the model knows what each
   voice sounds like. Here's the neat part: the audio is encoded to latents by the acoustic
   tokenizer, and those latents are spliced into the token sequence in place of runs of a
   placeholder token, `<|vision_pad|>` (a leftover vision token repurposed as "an audio embedding
   goes here"). A boolean `speech_input_mask` marks which positions are audio rather than text.
3. **The full script** — every turn formatted as a plain text line: `" Speaker 0: <text>\n"`,
   `" Speaker 1: <text>\n"`, and so on. Notably, there are *no special per-speaker tokens* — just
   these text labels. (That's why LongFlow's turn-localized steering has to track generated
   positions per turn rather than keying off special tokens.)

### 3.2 The autoregressive loop

Then the loop runs, once per output position. Most positions are **speech frames**; a few are
ordinary text-like tokens the model emits as punctuation for itself — in particular
`speech_start` / `speech_end` markers around each speaker's turn. For each speech frame:

1. **Think.** The backbone processes the newest input and produces a fresh hidden state —
   the `[B, 1536]` condition vector.
2. **Speak (in latent space).** The hidden state goes to the diffusion head, which denoises
   random noise into a 64-dim acoustic latent. This takes **10 denoising steps**, and each step
   runs the head **twice** for classifier-free guidance (section 4) — so 20 head passes per frame.
   (We pinned this down in P0: the model's default is 20 steps, but the shipped demo sets 10.)
3. **Render.** The σ-VAE decoder turns the latent into ~133 ms of actual 24 kHz audio.
4. **Listen to yourself.** The freshly decoded audio is **re-encoded** — the semantic tokenizer
   produces its 128-dim summary — and the acoustic and semantic features are passed through two
   small **connector** layers (Linear → RMSNorm → Linear) to build the input embedding for the
   next step.
5. Repeat until the script is spoken.

### 3.3 Why the feedback loop matters (LongFlow hard constraint 2)

Step 4 is easy to overlook and absolutely critical. The backbone never sees its own latents
directly — at every step it conditions on *the re-encoded version of the audio it just produced*.
Its hidden states are therefore shaped by that loop: encode → generate → decode → re-encode.

For LongFlow this is a hard constraint on training data: when we cache hidden states to train our
replacement flow head, those states **must come from real generation runs with the feedback loop
intact**. Hidden states computed any other way (say, teacher-forcing on ground-truth audio without
the loop) would be subtly different from what the head sees at inference time — and the head would
be trained on a distribution it never encounters.

### 3.4 The cost — and why LongFlow's C1 exists

Do the arithmetic for a 90-minute episode: **40,500 frames × 10 denoising steps × 2 CFG passes =
810,000 diffusion-head forward passes**, on top of 40,500 backbone steps. The head runs 20× more
often than the backbone. That's the target painted on its back: LongFlow's C1 replaces it with a
flow-matching head that needs 1–2 passes per frame and no CFG — a 10–20× reduction in head calls.

## 4. Classifier-free guidance (CFG), explained simply

**Classifier-free guidance** is a trick used all over diffusion models (it's how image generators
make pictures actually match the prompt). The idea in plain terms:

> Run the model twice. Once **with** the hints (the real conditioning — script, voices, context),
> and once **without** them (a blanked-out conditioning). The difference between the two
> predictions is *the part that came from the hints*. Now exaggerate exactly that part.

Concretely, at every denoising step the head produces a **positive** prediction `v_pos` (with the
real condition) and a **negative** prediction `v_neg` (with the empty condition), and combines
them:

```
v_guided = v_neg + cfg_scale * (v_pos - v_neg)
```

With `cfg_scale = 1` you'd just get `v_pos` back. VibeVoice's default is **`cfg_scale = 3.0`**
(we confirmed this live: the head's sampling function is
`sample_speech_tokens(condition, neg_condition, cfg_scale=3.0)`), so the condition's influence is
amplified 3×. The result: speech that sticks much more faithfully to the script and the reference
voices — at the price of running the head twice per step.

In [ ]:
# CFG on a toy 4-dim "prediction", just to see the arithmetic.
cfg_scale = 3.0  # VibeVoice's confirmed default

v_pos = torch.tensor([1.0, 0.2, -0.5, 0.0])   # prediction WITH the script/voice hints
v_neg = torch.tensor([0.4, 0.2, -0.1, 0.3])   # prediction with hints blanked out

v_guided = v_neg + cfg_scale * (v_pos - v_neg)

print("with hints    :", v_pos.tolist())
print("without hints :", v_neg.tolist())
print("guided (3x)   :", v_guided.tolist())
# Where the two agree (index 1), guidance changes nothing.
# Where they differ, the hint-driven part is pushed 3x harder.

### Our discovery: the CFG double stream inside the backbone

Here's the part you won't find in any paper — we found it empirically in P0 Stage 1
(`experiments/p0_steering/NOTES.md`).

Two pieces of vocabulary first. A **hook** is a small function you can attach to any PyTorch
module so it gets called every time that module runs — our way of watching (or editing) the
backbone's hidden states without touching its code. A **KV cache** is the transformer's memory of
positions it has already processed, so each new step only computes the newest position; the
**prefill** is the first big forward pass that processes the whole prompt at once and fills that
cache. Each call also carries a `cache_position` — the sequence index (or index range) being
processed right now.

The naive assumption: hook a decoder layer, generate N tokens, get 1 prefill call + N−1
incremental calls. Calibration falsified that. Generating **61 tokens**, our layer hook fired
**116 times**:

> **116 = 1 prefill + 60 positive-stream calls + 55 negative-stream calls**

What's going on: CFG's "without hints" prediction isn't free — the *backbone itself* maintains a
**parallel negative stream** with its own separate KV cache, and runs the same 28 layers a second
time to produce the negative condition. And it only does so on **speech-frame steps**: the 5
missing negative calls (60 − 55) correspond to steps that emitted text-like tokens (the
`speech_start`/`speech_end` turn markers), where no audio frame is generated and no CFG is needed.

Why this matters for LongFlow: our steering vectors must be extracted from (and injected into)
the **positive** stream only — polluting the negative stream would get our injection partially
*subtracted back out* by the CFG combination, or worse, distorted by the 3× amplification.

**The `cache_position` chain trick.** How do you tell the streams apart from inside a hook? The
two streams have separate KV caches, so each has its own position counter, and the counters
differ (the two prompts have different lengths). Each incremental call extends exactly one
stream's chain: its `cache_position` is that stream's last position + 1. So the recorder keeps
one "expected next position" per stream and matches each call to the chain it continues. The
cell below simulates the whole thing with plain ints.

In [ ]:
# Simulate classifying hook calls into streams via cache_position chains.
# Numbers mirror the real P0 calibration: 61 tokens -> 116 hook calls.

pos_prompt_len = 50   # positive stream: full prompt (system + voices + script)
neg_prompt_len = 30   # negative stream: blanked prompt -> shorter, so chains never collide

# 60 generation steps; 55 are speech frames (True), 5 are text/marker tokens (False).
is_speech_frame = [True]*20 + [False] + [True]*20 + [False]*4 + [True]*15
assert len(is_speech_frame) == 60 and sum(is_speech_frame) == 55

# --- Build the call sequence a layer hook would observe -------------------
calls = [list(range(pos_prompt_len))]          # prefill: a whole RANGE of positions
p, n = pos_prompt_len, neg_prompt_len
for speech in is_speech_frame:
    calls.append([p]); p += 1                  # positive pass: every step
    if speech:
        calls.append([n]); n += 1              # negative pass: speech frames only

# --- The recorder's classification logic ----------------------------------
expected_next = {"positive": pos_prompt_len, "negative": neg_prompt_len}
counts = {"prefill": 0, "positive": 0, "negative": 0}

for cache_position in calls:
    if len(cache_position) > 1:                # many positions at once = prefill
        counts["prefill"] += 1
        continue
    pos = cache_position[0]
    for stream in ("positive", "negative"):    # which chain does this call extend?
        if pos == expected_next[stream]:
            counts[stream] += 1
            expected_next[stream] = pos + 1
            break

print(counts)
print("total hook calls:", sum(counts.values()))
assert counts == {"prefill": 1, "positive": 60, "negative": 55}
assert sum(counts.values()) == 116   # the real calibration number

## 5. Why everything stays frozen — and where LongFlow hooks in

**The one-paragraph N1/N2 story.** LongFlow's predecessor, TransplantTTS, tried to train a *new*
backbone (Qwen3) to produce conditioning states for these same tokenizers, pre-training it with a
simple MSE regression loss on the latents. Result N1: that objective locked the backbone's hidden
states into "mean-prediction-shaped" features — a discrete classification head on top plateaued at
~32% accuracy, and 75× more data didn't help. Result N2 was the killer: the *exact* flow-matching
head LongFlow plans to use (15M params, AdaLN MLP, no attention) was trained on those
MSE-shaped states, and produced non-speech output — the architecture was fine, the *conditioning
signal* was starved. The lesson: a generative head is only as good as the hidden states feeding
it. VibeVoice's own Qwen2.5 states are **pre-validated** conditioning — every one of them already
produced intelligible speech through the original diffusion head. So LongFlow trains **nothing**
upstream: no backbone training, no tokenizer training. Only the small head is new; steering and
anchoring are pure inference-time interventions.

**Where our code attaches** (the verified hook map, `docs/resources.md` §1):

- **Steering** hooks the 28 decoder layers at `model.model.language_model.layers[i]`, using the
  cache-position chains from section 4 to touch only the positive stream, turn-localized per
  speaker.
- **Flow-head training data** is captured at **`sample_speech_tokens(condition, neg_condition,
  cfg_scale=3.0)`** — and this is the *perfect* capture point. It is the single function through
  which every speech frame flows, and its arguments are exactly the training pair we need: the
  `[B, 1536]` positive condition (the pre-validated thought) alongside the 64-dim latent it goes
  on to produce. One hook there yields (hidden state, latent) pairs straight from the real
  generation loop — feedback loop included, satisfying hard constraint 2 for free. No stream
  disambiguation needed either, since positive and negative conditions arrive as separate,
  labeled arguments.

One honest footnote: P0 also taught us the *limits* of the frozen backbone. Activation steering
worked mechanically (clean turn localization, zero WER damage in the usable window) but the
perceptual emotion shift capped out at "subtle" — VibeVoice is a stability-first model with
little affect dynamic range, and steering can't exceed the range the backbone exposes (N7). The
paper proceeds on the flow head, speaker anchoring, and the consistency benchmark.

## 6. Go deeper

- **VibeVoice technical report** — the primary source for the architecture:
  [arXiv:2508.19205](https://arxiv.org/abs/2508.19205)
- **LatentLM: "Multimodal Latent Language Modeling with Next-Token Diffusion"** — where the
  next-token-diffusion idea and the σ-VAE come from:
  [arXiv:2412.08635](https://arxiv.org/abs/2412.08635)
- **The community code fork LongFlow pins** (Microsoft's original repo went quiet; this is the
  maintained mirror, MIT): [github.com/vibevoice-community/VibeVoice](https://github.com/vibevoice-community/VibeVoice)
  — the files referenced above live under `vibevoice/modular/` and `vibevoice/processor/`.
- **The weights**: [huggingface.co/microsoft/VibeVoice-1.5B](https://huggingface.co/microsoft/VibeVoice-1.5B)
- **Qwen2.5 technical report** — the backbone family:
  [arXiv:2412.15115](https://arxiv.org/abs/2412.15115)
- **Eren Gölge's VibeVoice analysis** (author of Coqui TTS; a sharp independent read of the
  architecture, posted 2025-08-26): [erogol.com](https://erogol.com) — look for the VibeVoice
  post dated 2025/08/26.
- **This repo's ground truth**: `docs/resources.md` (verified hook map),
  `experiments/p0_steering/NOTES.md` (the double-stream discovery and the full P0 record),
  `docs/negative-results.md` (N1–N7 — why the design is what it is).